<a href="https://colab.research.google.com/github/nbvhung/identifyApplications_AI/blob/main/dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#ô code 1
import pandas as pd

from google.colab import drive
import pandas as pd
import os

# 1. Kết nối Drive
drive.mount('/content/drive')

# 2. Đường dẫn "bất tử" - Không bao giờ phải sửa lại
path = '/content/drive/MyDrive/datasetAI/Dataset-Unicauca-Version2-87Atts.csv'

# 3. Đọc dữ liệu
if os.path.exists(path):
    df = pd.read_csv(path)
    print(f"Tổng số dòng: {len(df)}")
df = df.sample(100000, random_state=42).reset_index(drop=True) #lấy 100k dòng chạy thử và đánh lại số tt chạy từ đầu
# Lệnh này sẽ gom nhóm và đếm tất cả các loại app có trong file
print(df['ProtocolName'].unique())
print(f"Tổng số ứng dụng khác nhau là: {df['ProtocolName'].nunique()}")



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Tổng số dòng: 3577296
['HTTP' 'GOOGLE' 'HTTP_PROXY' 'HTTP_CONNECT' 'MICROSOFT' 'AMAZON'
 'CONTENT_FLASH' 'YOUTUBE' 'GMAIL' 'SSL' 'WINDOWS_UPDATE' 'FACEBOOK' 'MSN'
 'SKYPE' 'DROPBOX' 'OFFICE_365' 'TWITTER' 'CLOUDFLARE' 'TEAMVIEWER'
 'YAHOO' 'NETFLIX' 'IP_ICMP' 'DNS' 'SPOTIFY' 'MS_ONE_DRIVE' 'WHATSAPP'
 'APPLE' 'UBUNTUONE' 'APPLE_ITUNES' 'GOOGLE_MAPS' 'EBAY' 'SSL_NO_CERT'
 'INSTAGRAM' 'MQTT' 'TIMMEU' 'FTP_DATA' 'WAZE' 'H323' 'WIKIPEDIA'
 'APPLE_ICLOUD' 'EASYTAXI' 'RADIUS' 'EDONKEY' 'HTTP_DOWNLOAD' 'DEEZER'
 'TOR' 'NTP' 'FTP_CONTROL' 'WHOIS_DAS' 'OSCAR' 'TELEGRAM' 'SSH' 'ORACLE'
 'BGP']
Tổng số ứng dụng khác nhau là: 54


In [2]:
#ô code 2
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
import numpy as np

# 1. Mã hóa nhãn: Biến 'YOUTUBE', 'FACEBOOK'... thành số 0, 1, 2...
le = LabelEncoder()
df['ProtocolName_Encoded'] = le.fit_transform(df['ProtocolName'])

# 2. Chọn các cột đặc trưng (Features) - Loại bỏ các cột không phải số hoặc không cần thiết
# Mình tạm loại bỏ các cột IP và ID vì chúng dễ làm AI bị "học vẹt"
features = df.select_dtypes(include=[np.number]).columns.tolist()
features = [f for f in features if f not in ['ProtocolName_Encoded', 'L7Protocol']]

X = df[features]
y = df['ProtocolName_Encoded']

# 3. Xử lý giá trị vô hạn (Inf) hoặc lỗi dữ liệu thường gặp trong lưu lượng mạng
X = X.replace([np.inf, -np.inf], np.nan).fillna(0)

# 4. Chia dữ liệu: 80% để học, 20% để kiểm tra
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 5. Chuẩn hóa: Đưa các con số về cùng một thang đo (0-1 hoặc tương đương)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Đã chuẩn bị xong dữ liệu cho {len(features)} đặc trưng và {df['ProtocolName'].nunique()} ứng dụng.")


Đã chuẩn bị xong dữ liệu cho 80 đặc trưng và 54 ứng dụng.


In [3]:
#ô code 3
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import LabelEncoder

# 1. Tái mã hóa nhãn riêng cho tập Train và Test để đảm bảo tính liên tục (0, 1, 2...)
# Bước này cực kỳ quan trọng để sửa lỗi "Invalid classes inferred"
le_final = LabelEncoder()
y_train_fixed = le_final.fit_transform(y_train)

# Đối với tập Test, chúng ta chỉ lấy những nhãn mà tập Train đã học được
# Những nhãn nào tập Train không có sẽ bị loại bỏ ở tập Test để không gây lỗi
test_mask = y_test.isin(le_final.classes_)
X_test_fixed = X_test_scaled[test_mask]
y_test_fixed = le_final.transform(y_test[test_mask])

# 2. Khởi tạo mô hình XGBoost
model_baseline = XGBClassifier(n_estimators=100, random_state=42, n_jobs=-1)

# 3. Huấn luyện mô hình
print(f"Đang huấn luyện mô hình cơ sở trên {len(le_final.classes_)} ứng dụng...")
model_baseline.fit(X_train_scaled, y_train_fixed)

# 4. Dự đoán và đánh giá
y_pred = model_baseline.predict(X_test_fixed)
print(f"\n--- KẾT QUẢ HOÀN THÀNH WEEK 3 ---")
print(f"Độ chính xác (Accuracy): {accuracy_score(y_test_fixed, y_pred):.4f}")
print("\nBáo cáo chi tiết (Classification Report):")
print(classification_report(y_test_fixed, y_pred))

Đang huấn luyện mô hình cơ sở trên 54 ứng dụng...

--- KẾT QUẢ HOÀN THÀNH WEEK 3 ---
Độ chính xác (Accuracy): 0.7328

Báo cáo chi tiết (Classification Report):
              precision    recall  f1-score   support

           0       0.75      0.47      0.57       496
           1       0.81      0.39      0.53        54
           2       0.00      0.00      0.00         5
           3       0.00      0.00      0.00        11
           5       0.70      0.37      0.48        82
           6       0.79      0.68      0.73        40
           7       0.00      0.00      0.00         1
           8       0.67      0.50      0.57         4
           9       0.95      0.69      0.80       157
          10       1.00      0.17      0.29         6
          11       0.00      0.00      0.00         7
          13       0.87      0.75      0.80       164
          15       0.00      0.00      0.00         2
          16       0.37      0.09      0.14       222
          17       0.71      

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [4]:
#
#ô code 4
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout
from tensorflow.keras.layers import BatchNormalization

In [5]:
# ô code 5
import numpy as np
# Chuyển từ 2D sang 3D bằng cách dùng đúng tên biến đã chuẩn hóa ở trên
X_train_cnn = np.expand_dims(X_train_scaled, axis=2)
X_test_cnn = np.expand_dims(X_test_fixed, axis=2)

print(f"Cấu trúc dữ liệu 3D cho CNN: {X_train_cnn.shape}")

Cấu trúc dữ liệu 3D cho CNN: (80000, 80, 1)


In [6]:
# ô code 6
num_features = X_train_scaled.shape[1] # Tự động lấy số lượng cột (đặc trưng)
num_classes = len(le_final.classes_)   # Tự động lấy số lượng ứng dụng

model = Sequential([
    # Lớp lọc đặc trưng 1(Convolutional Layer)
    Conv1D(64, 3, activation='relu', input_shape=(num_features, 1)),
    BatchNormalization(), #chuẩn hóa theo lô (giúp dữ liệu ổn định hơn)
    MaxPooling1D(2),
    #Lớp lọc đặc trưng 2
    Conv1D(128, 3, activation='relu', input_shape=(num_features, 1)),
    BatchNormalization(),
    MaxPooling1D(2),
    Dropout(0.2), #giảm 20% neuron để tránh học vẹt
    # Lớp làm phẳng dữ liệu để đưa vào mạng nơ-ron
    Flatten(),
    Dense(256, activation='relu'), #256 neuron
    Dropout(0.4), # drop thêm neuron để dữ liệu chạy ổn định
    # Lớp đầu ra (Số lượng app thực tế)
    Dense(num_classes, activation='softmax') # Khớp với số lượng app thực tế
])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 78, 64)         │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 78, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 39, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 37, 128)        │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 37, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 18, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 18, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 2304)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 54)             │        13,878 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 629,686 (2.40 MB)

 Trainable params: 629,302 (2.40 MB)

 Non-trainable params: 384 (1.50 KB)

In [8]:
# ô code 7
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# 1. Tự động dừng nếu 5 vòng không tăng accuracy (tiết kiệm thời gian cho Mạnh)
early_stop = EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True)

# 2. Tự động giảm tốc độ học nếu bị "kẹt" (giúp mô hình học sâu hơn)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3)
# Sử dụng y_train_fixed và y_test_fixed để khớp với mã hóa của XGBoost ở trên
history = model.fit(
    X_train_cnn,
    y_train_fixed,
    epochs=50,
    batch_size=128,
    validation_data=(X_test_cnn, y_test_fixed),
    callbacks=[early_stop, reduce_lr] # hàm gọi lại khi hết 1 epoch git kiểm tra mô hình có tốt lên ko ? có nên dùng ko
)

Epoch 1/50
625/625 ━━━━━━━━━━━━━━━━━━━━ 11s 8ms/step - accuracy: 0.4779 - loss: 1.7137 - val_accuracy: 0.5286 - val_loss: 1.4534 - learning_rate: 0.0010
Epoch 2/50
625/625 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.5326 - loss: 1.4886 - val_accuracy: 0.5638 - val_loss: 1.3666 - learning_rate: 0.0010
Epoch 3/50
625/625 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.5523 - loss: 1.4161 - val_accuracy: 0.5777 - val_loss: 1.3069 - learning_rate: 0.0010
Epoch 4/50
625/625 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.5693 - loss: 1.3603 - val_accuracy: 0.5968 - val_loss: 1.2643 - learning_rate: 0.0010
Epoch 5/50
625/625 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.5781 - loss: 1.3253 - val_accuracy: 0.5906 - val_loss: 1.2556 - learning_rate: 0.0010
Epoch 6/50
625/625 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - accuracy: 0.5840 - loss: 1.2982 - val_accuracy: 0.6040 - val_loss: 1.2211 - learning_rate: 0.0010
Epoch 7/50
625/625 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.5911 - loss: 1.2767 -

In [11]:
from sklearn.metrics import classification_report
import numpy as np

# 1. Dự đoán lớp
y_pred = model.predict(X_test_cnn)
y_pred_classes = np.argmax(y_pred, axis=1)

# 2. Lấy các nhãn thực tế xuất hiện (tránh lỗi 40 vs 54)
present_labels = np.unique(np.concatenate([y_test_fixed, y_pred_classes]))

# 3. ÉP KIỂU SANG CHUỖI (Sửa lỗi TypeError ở ảnh f765ee)
present_names = [str(le_final.classes_[i]) for i in present_labels]

# 4. In báo cáo chi tiết
print(classification_report(y_test_fixed, y_pred_classes,
                            labels=present_labels,
                            target_names=present_names))

625/625 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step
              precision    recall  f1-score   support

           0       0.84      0.34      0.48       496
           1       0.88      0.13      0.23        54
           2       0.00      0.00      0.00         5
           3       0.00      0.00      0.00        11
           5       0.74      0.30      0.43        82
           6       0.88      0.70      0.78        40
           7       0.00      0.00      0.00         1
           8       0.57      1.00      0.73         4
           9       0.93      0.60      0.73       157
          10       0.00      0.00      0.00         6
          11       0.00      0.00      0.00         7
          13       0.81      0.70      0.75       164
          15       0.00      0.00      0.00         2
          16       0.88      0.03      0.06       222
          17       0.64      0.78      0.70      5391
          18       0.00      0.00      0.00         8
          20       0.75      0.89      0

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
